# LLM Steganography — Benchmarki (Kaggle, **online**)

Pliki projektu: `/kaggle/input/datasets/kamilml/sources/`
Wyniki i cache: `/kaggle/working/Steganography_colab/`

Wymaga: internet + secret `HF_TOKEN`.

> Offline: `Kaggle_Runner_Offline.ipynb`

In [2]:
import os
import shutil
from pathlib import Path

INPUT_DIR = Path('/kaggle/input/datasets/kamilml/sources1')
PROJECT_DIR = Path('/kaggle/working/Steganography_colab')
MODEL_CACHE_DIR = PROJECT_DIR / 'models_cache'
REQUIRED = ['run_experiments.py', 'dummy_processor.py', 'requirements.txt']

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

missing = [name for name in REQUIRED if not (INPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Brakuje w {INPUT_DIR}: {missing}')

for name in REQUIRED:
    shutil.copy2(INPUT_DIR / name, PROJECT_DIR / name)
    print(f'copied {name}')

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = str(MODEL_CACHE_DIR)

print('cwd:', Path.cwd())
print('input:', INPUT_DIR)
print('model cache:', MODEL_CACHE_DIR)


copied run_experiments.py
copied dummy_processor.py
copied requirements.txt
cwd: /kaggle/working/Steganography_colab
input: /kaggle/input/datasets/kamilml/sources1
model cache: /kaggle/working/Steganography_colab/models_cache


In [3]:
%%writefile humaneval_subset_eval.py
"""HumanEval pass@1 for arbitrary task subsets (no 164-task assert)."""

from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any, Callable

from human_eval.execution import check_correctness


def evaluate_pass_at_1_subset(
    problems: list[dict[str, str]],
    predictions: list[str],
    extract_code_fn: Callable[[str], str],
    *,
    timeout: float = 3.0,
    n_workers: int = 4,
) -> dict[str, Any]:
    if len(problems) != len(predictions):
        raise ValueError(
            f"predictions ({len(predictions)}) must match problems ({len(problems)})"
        )

    samples = [
        {
            "task_id": problem["task_id"],
            "completion": extract_code_fn(prediction),
        }
        for problem, prediction in zip(problems, predictions)
    ]

    task_results: list[dict[str, Any]] = []
    order = {problem["task_id"]: idx for idx, problem in enumerate(problems)}

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        future_map = {
            executor.submit(
                check_correctness,
                problem,
                sample["completion"],
                timeout,
                completion_id,
            ): (problem["task_id"], sample["completion"])
            for completion_id, (problem, sample) in enumerate(zip(problems, samples))
        }
        for future in as_completed(future_map):
            task_id, completion = future_map[future]
            row = future.result()
            task_results.append(
                {
                    "task_id": task_id,
                    "passed": bool(row.get("passed", False)),
                    "result": row.get("result", ""),
                    "completion": completion,
                }
            )

    task_results.sort(key=lambda row: order[row["task_id"]])
    passed_count = sum(1 for row in task_results if row["passed"])
    total = len(task_results)

    return {
        "pass_at_1": passed_count / total if total else 0.0,
        "task_results": task_results,
        "passed_count": passed_count,
        "failed_count": total - passed_count,
        "total_count": total,
    }


Writing humaneval_subset_eval.py


In [4]:
# Auto-łatka: stary run_experiments.py z input → nowa ewaluacja HumanEval
import re
from pathlib import Path

he = Path('humaneval_subset_eval.py')
if not he.exists():
    raise FileNotFoundError('Uruchom komórkę %%writefile humaneval_subset_eval.py')

rp = Path('run_experiments.py')
text = rp.read_text(encoding='utf-8')

if 'humaneval_subset_eval' in text and 'evaluate_pass_at_1_subset' in text:
    print('run_experiments.py już OK')
else:
    if 'from humaneval_subset_eval import evaluate_pass_at_1_subset' not in text:
        text = text.replace(
            'from dummy_processor import',
            'from humaneval_subset_eval import evaluate_pass_at_1_subset\nfrom dummy_processor import',
            1,
        )
    new_fn = """def evaluate_pass_at_1(
    problems: list[dict[str, str]],
    predictions: list[str],
    *,
    timeout: float = 3.0,
    n_workers: int = 4,
) -> dict[str, Any]:
    return evaluate_pass_at_1_subset(
        problems,
        predictions,
        extract_code_for_eval,
        timeout=timeout,
        n_workers=n_workers,
    )
"""
    text, n = re.subn(
        r"def evaluate_pass_at_1\(.*?)(?=\ndef [a-z_]|\nclass )",
        new_fn + "\n",
        text,
        count=1,
        flags=re.DOTALL,
    )
    if n != 1:
        raise RuntimeError('Nie udało się załatać evaluate_pass_at_1 — wgraj nowy run_experiments.py do input')
    rp.write_text(text, encoding='utf-8')
    print('Załatano run_experiments.py')

if 'evaluate_functional_correctness' in rp.read_text(encoding='utf-8'):
    print('WARN: nadal jest evaluate_functional_correctness w pliku')
else:
    print('Weryfikacja OK')


run_experiments.py już OK
Weryfikacja OK


In [5]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.3 MB/s eta 0:00:00


In [6]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = os.environ.get('HF_TOKEN', '').strip()
if not hf_token:
    secrets = UserSecretsClient()
    for key in ('HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN'):
        try:
            hf_token = secrets.get_secret(key)
            print(f'Secret: {key}')
            break
        except Exception:
            pass

if not hf_token:
    raise RuntimeError('Dodaj secret HF_TOKEN w Kaggle')

os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
print('HF login OK')

Secret: HF_TOKEN


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK


## Konfiguracja przebiegu

In [7]:
TEST = 'humaneval'
MODEL = 'gemma'
THRESHOLD = 'all'
TOP_N = 16
MAX_NEW_TOKENS = 512
HUMANEVAL_TASKS = None
NO_MODEL_CACHE = True

In [8]:
_humaneval_args = f' --humaneval-tasks {HUMANEVAL_TASKS}' if HUMANEVAL_TASKS else ''
_no_cache_args = ' --no-model-cache' if NO_MODEL_CACHE else ''

!python run_experiments.py \
  --test {TEST} \
  --model {MODEL} \
  --threshold {THRESHOLD} \
  --top-n {TOP_N} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --model-cache-dir {MODEL_CACHE_DIR}{_humaneval_args}{_no_cache_args}

Model cache disabled — temporary HF dir: /tmp/stego_hf_nocache_6r1_aq52
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Logged in to Hugging Face from environment token.

=== Running test=humaneval | model=gemma (google/gemma-2-9b-it) | threshold=0.0 | top_n=16 ===
Loading from Hugging Face Hub (no persistent cache): google/gemma-2-9b-it
config.json: 100%|█████████████████████████████| 857/857 [00:00<00:00, 2.98MB/s]
tokenizer_config.json: 47.0kB [00:00, 67.4MB/s]
tokenizer.json: 100%|██████████████████████| 17.5M/17.5M [00:00<00:00, 22.9MB/s]
special_tokens_map.json: 100%|█████████████████| 636/636 [00:00<00:00, 3.36MB/s]
model.safetensors.index.json: 39.1kB [00:00, 77.6MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:12<00:00, 18.04s/it]
Download complete: 100%|████████████████████| 18.5G/18.5G [01:12<00:00, 256MB/s]
Loading weights: 100%|█| 464/464 [00:39<00:00, 11.67it/s, Materializing

In [ ]:
import csv

summary_csv = Path('results/summary.csv')
if summary_csv.exists():
    with summary_csv.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'Rows in summary.csv: {len(rows)}')
    for row in rows[-5:]:
        print(row)
else:
    print('summary.csv not found yet')